In [1]:
!pip install vaderSentiment

Defaulting to user installation because normal site-packages is not writeable


#Load reviews fresh

In [1]:
import pandas as pd

reviews = pd.read_parquet("../data/interim/reviews_silver.parquet")
listings = pd.read_parquet("../data/interim/listings_final2_silver.parquet")
print(reviews.shape, listings.shape)

(990170, 6) (30259, 95)


#Run sentiment analysis

In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Drop rows with missing comments first (272 missing, per §3.1)
reviews_clean = reviews.dropna(subset=['comments']).copy()

def get_sentiment(text):
    return analyzer.polarity_scores(str(text))['compound']

# Sample for speed if needed — 990K reviews may take a few minutes on full data
sample_reviews = reviews_clean.sample(min(50000, len(reviews_clean)), random_state=42)
sample_reviews['sentiment'] = sample_reviews['comments'].apply(get_sentiment)

print(sample_reviews['sentiment'].describe())

count    50000.000000
mean         0.718332
std          0.386132
min         -0.996400
25%          0.658800
50%          0.888300
75%          0.956400
max          0.999600
Name: sentiment, dtype: float64


#Correlate sentiment with numerical review scores

In [3]:
sentiment_by_listing = sample_reviews.groupby('listing_id')['sentiment'].mean().reset_index()
sentiment_by_listing.columns = ['id', 'avg_sentiment']

merged = listings[['id', 'review_scores_rating']].merge(sentiment_by_listing, on='id', how='inner')
print(f"Listings with both sentiment and rating: {len(merged)}")

correlation = merged[['avg_sentiment', 'review_scores_rating']].corr().iloc[0,1]
print(f"Correlation between review text sentiment and numerical rating: {correlation:.3f}")

Listings with both sentiment and rating: 11339
Correlation between review text sentiment and numerical rating: 0.270


In [4]:
merged_with_text = sample_reviews.merge(listings[['id','review_scores_rating']], left_on='listing_id', right_on='id')
mismatch = merged_with_text[
    (merged_with_text['sentiment'] < 0.3) & (merged_with_text['review_scores_rating'] >= 4.8)
][['comments', 'sentiment', 'review_scores_rating']]

print(f"Reviews with low text sentiment but high numerical rating: {len(mismatch)}")
print(mismatch.head(3))

Reviews with low text sentiment but high numerical rating: 2754
                                             comments  sentiment  \
5   Buenas: Subtes y transportes a 300mts, Superme...        0.0   
28                                      Todo muy bien        0.0   
38                                       所有的都很好，无可挑剔。        0.0   

    review_scores_rating  
5                   4.89  
28                  4.96  
38                  4.89  
